# 🇮🇳 Hindi Language Ambiguity Detection using mBERT
### NLP Assignment — Transformer Model to Handle Ambiguity in Hindi

**Task:** Classify Hindi sentences into three types of linguistic ambiguity:
- 🔤 **Lexical** — A word has multiple meanings (e.g., आम = mango / common)
- 🌳 **Syntactic** — Sentence structure allows multiple parse trees
- 🔬 **Semantic** — Sentence meaning is logically ambiguous

**Model:** `bert-base-multilingual-cased` (mBERT) — trained on 104 languages including Hindi


## 📦 Step 1 — Install & Import Dependencies

In [1]:
# Install required libraries
!pip install transformers datasets torch scikit-learn seaborn torch matplotlib pandas tqdm --quiet
print('✅ All libraries installed!')

✅ All libraries installed!



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Bigboy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from transformers import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
from tqdm import tqdm

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔧 Using device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

ModuleNotFoundError: No module named 'torch'

## 📚 Step 2 — Hindi Ambiguity Dataset

We use a curated dataset of Hindi sentences annotated with three ambiguity types.
The dataset covers:
- Common Hindi homonyms and polysemous words (Lexical)
- PP-attachment and coordination ambiguities (Syntactic)
- Quantifier scope and negation scope ambiguities (Semantic)

**Labels:** `0 = Lexical`, `1 = Syntactic`, `2 = Semantic`

In [ ]:
# ============================================================
# HINDI AMBIGUITY DATASET
# Label 0 = Lexical Ambiguity
# Label 1 = Syntactic Ambiguity
# Label 2 = Semantic Ambiguity
# ============================================================

raw_data = [
    # ─── LEXICAL AMBIGUITY (Label 0) ───────────────────────────
    # 'आम' = mango OR common
    {"text": "मुझे आम खाना है।", "label": 0, "note": "'आम' = mango / common"},
    {"text": "आम आदमी की बात करो।", "label": 0, "note": "'आम' = common / ordinary"},
    {"text": "यह एक आम समस्या है।", "label": 0, "note": "'आम' = common"},
    # 'काला' = black OR art/skill
    {"text": "उसका काला जादू प्रसिद्ध है।", "label": 0, "note": "'काला' = black / dark arts"},
    {"text": "वह काला कपड़ा पहनता है।", "label": 0, "note": "'काला' = black (color)"},
    # 'काल' = time/era OR death
    {"text": "यह बुरा काल है।", "label": 0, "note": "'काल' = era / death"},
    {"text": "काल आएगा तो देखेंगे।", "label": 0, "note": "'काल' = tomorrow / death"},
    # 'पत्ता' = leaf OR card/clue
    {"text": "उसने पत्ता फेंका।", "label": 0, "note": "'पत्ता' = leaf / playing card"},
    {"text": "मुझे पत्ता नहीं था।", "label": 0, "note": "'पत्ता' = knowledge/clue"},
    # 'बाल' = hair OR child
    {"text": "उसके बाल बहुत सुंदर हैं।", "label": 0, "note": "'बाल' = hair / child"},
    {"text": "बाल श्रम बंद होना चाहिए।", "label": 0, "note": "'बाल' = child"},
    # 'पानी' = water OR prestige
    {"text": "उसका पानी उतर गया।", "label": 0, "note": "'पानी' = water / prestige"},
    # 'कान' = ear
    {"text": "उसने कान पर हाथ रखा।", "label": 0, "note": "'कान' = ear / attention"},
    # 'खेल' = game OR play
    {"text": "यह तो बड़ा खेल है।", "label": 0, "note": "'खेल' = game / scheme"},
    # 'सर' = head OR sir
    {"text": "सर ने बुलाया है।", "label": 0, "note": "'सर' = sir / head"},
    {"text": "उसका सर दर्द कर रहा है।", "label": 0, "note": "'सर' = head"},
    # 'नेता' = leader / politician
    {"text": "वह एक बड़ा नेता है।", "label": 0, "note": "'नेता' = leader / politician"},
    # 'भारी' = heavy / serious
    {"text": "यह बहुत भारी काम है।", "label": 0, "note": "'भारी' = heavy / difficult"},
    # 'हाथ' = hand / help
    {"text": "उसका हाथ पकड़ो।", "label": 0, "note": "'हाथ' = hand / support"},
    # 'मुँह' = mouth / face
    {"text": "उसने मुँह फेर लिया।", "label": 0, "note": "'मुँह' = face / expression"},
    # 'दिल' = heart / feelings
    {"text": "मेरा दिल नहीं लगता।", "label": 0, "note": "'दिल' = heart / mind"},
    # 'आँख' = eye / vision
    {"text": "उस पर आँख रखो।", "label": 0, "note": "'आँख' = eye / surveillance"},
    # 'गर्म' = hot / angry
    {"text": "वह बहुत गर्म मिज़ाज का है।", "label": 0, "note": "'गर्म' = hot / short-tempered"},
    # 'ठंडा' = cold / calm
    {"text": "उसने ठंडा जवाब दिया।", "label": 0, "note": "'ठंडा' = cold / indifferent"},
    # 'रोशनी' = light / clarity
    {"text": "इस मामले पर रोशनी डालो।", "label": 0, "note": "'रोशनी' = light / insight"},
    # 'नमक' = salt / loyalty
    {"text": "उसने नमक का हक अदा किया।", "label": 0, "note": "'नमक' = salt / loyalty"},
    # 'छाया' = shadow / protection
    {"text": "उनकी छाया में रहो।", "label": 0, "note": "'छाया' = shadow / shelter"},
    # 'रास्ता' = road / way/method
    {"text": "कोई रास्ता निकालो।", "label": 0, "note": "'रास्ता' = road / solution"},
    # 'घर' = house / family
    {"text": "घर की बात घर में रहे।", "label": 0, "note": "'घर' = house / family"},
    # 'मिठास' = sweetness / kindness
    {"text": "उसकी बातों में मिठास है।", "label": 0, "note": "'मिठास' = sweetness / warmth"},

    # ─── SYNTACTIC AMBIGUITY (Label 1) ─────────────────────────
    # PP-attachment ambiguity
    {"text": "उसने लाठी लेकर आदमी को देखा।", "label": 1, "note": "PP: who had the stick?"},
    {"text": "मैंने दूरबीन से आदमी को देखा।", "label": 1, "note": "PP: who had the binoculars?"},
    {"text": "लड़की ने छत पर बैठे आदमी को देखा।", "label": 1, "note": "PP: who was on roof?"},
    {"text": "राम ने चाकू से केक काटा और सोहन ने भी।", "label": 1, "note": "Coordination ambiguity"},
    {"text": "वह बुजुर्ग महिला और बच्चे को देखकर रो पड़ा।", "label": 1, "note": "NP scope: who triggered emotion?"},
    {"text": "मैंने शेर को पिंजरे में देखा।", "label": 1, "note": "PP: who was in cage?"},
    {"text": "वह सफेद कपड़े पहने लड़की का दोस्त है।", "label": 1, "note": "Modifier attachment"},
    {"text": "उसने रोते हुए बच्चे को दूध दिया।", "label": 1, "note": "Participial phrase attachment"},
    {"text": "राधा और कृष्ण के भक्त आए।", "label": 1, "note": "NP coordination scope"},
    {"text": "मोटे आदमियों और बच्चों का रेस्तरां बंद है।", "label": 1, "note": "Adjective scope over coordination"},
    # Free word-order ambiguity
    {"text": "राम श्याम को पसंद करता है।", "label": 1, "note": "SOV order: who likes whom?"},
    {"text": "मोहन को सोहन ने मारा या रोहन ने?", "label": 1, "note": "Subject ambiguity"},
    {"text": "अध्यापक ने छात्रों को किताबें दिखाईं जो नई थीं।", "label": 1, "note": "Relative clause attachment"},
    # Conjunction ambiguity
    {"text": "वह गाना और नाचना पसंद करता है और वह भी।", "label": 1, "note": "Ellipsis + coordination"},
    {"text": "रीता ने सुनीता को सफल गायिका और नर्तकी कहा।", "label": 1, "note": "Predicate coordination scope"},
    {"text": "बड़े शहरों और गाँवों के लोग आए।", "label": 1, "note": "Adjective scope"},
    {"text": "उसने कपड़े धोए और बर्तन भी।", "label": 1, "note": "Ellipsis ambiguity"},
    {"text": "मैंने अपने भाई के साथ उसके दोस्त को देखा।", "label": 1, "note": "PP attachment / pronoun ref"},
    {"text": "वह लड़की जो कल आई थी उसकी बहन है।", "label": 1, "note": "Relative clause attachment"},
    {"text": "नीली साड़ी और कुर्ते वाली महिला मेरी माँ है।", "label": 1, "note": "Adjective scope over NP"},
    # More syntactic
    {"text": "मैं बच्चों को पढ़ाने वाले शिक्षक से मिला।", "label": 1, "note": "Relative clause"},
    {"text": "राम ने टोपी पहने बच्चे को बुलाया।", "label": 1, "note": "Participial modifier scope"},
    {"text": "सेब और आम खाने वाले बच्चे बीमार हैं।", "label": 1, "note": "Coordination scope"},
    {"text": "उसने दौड़ते हुए घोड़े को पकड़ा।", "label": 1, "note": "Who was running?"},
    {"text": "वह बड़े परिवार और बच्चों का ख्याल रखता है।", "label": 1, "note": "NP scope"},
    {"text": "उस कमरे में एक आदमी और एक औरत के साथ बच्चा था।", "label": 1, "note": "Comitative ambiguity"},
    {"text": "मेरे बगल में खड़े आदमी की पत्नी डॉक्टर है।", "label": 1, "note": "Modifier attachment"},
    {"text": "उसने गाना सुनते हुए पढ़ाई की।", "label": 1, "note": "Adverbial clause scope"},
    {"text": "नाचती हुई लड़की को देखकर वह हँसा।", "label": 1, "note": "Participial attachment"},
    {"text": "उसने अपनी माँ के साथ बाज़ार जाने से मना किया।", "label": 1, "note": "Complement scope"},

    # ─── SEMANTIC AMBIGUITY (Label 2) ──────────────────────────
    # Scope of quantifiers / negation
    {"text": "हर छात्र ने कोई किताब नहीं पढ़ी।", "label": 2, "note": "Quantifier scope: ∀∃ or ∃∀?"},
    {"text": "हर लड़की एक लड़के से प्यार करती है।", "label": 2, "note": "Quantifier scope ambiguity"},
    {"text": "सभी बच्चे किसी न किसी खेल को पसंद करते हैं।", "label": 2, "note": "Scope: universal vs existential"},
    {"text": "वह नहीं आएगी क्योंकि वह बीमार है।", "label": 2, "note": "Causal scope ambiguity"},
    # Pronoun reference ambiguity (coreference)
    {"text": "रमेश ने सुरेश को बताया कि वह पास हो गया।", "label": 2, "note": "Pronoun 'वह' = Ramesh or Suresh?"},
    {"text": "सीता ने गीता से कहा कि वह बहुत चालाक है।", "label": 2, "note": "'वह' = Sita or Gita?"},
    {"text": "अमित और विकास दोनों गए और वह लौटा।", "label": 2, "note": "Coreference: 'वह' = whom?"},
    {"text": "माँ ने बेटी से कहा कि उसे डॉक्टर बनना है।", "label": 2, "note": "'उसे' = mother or daughter?"},
    # Presupposition / implicature
    {"text": "क्या तुमने झूठ बोलना बंद कर दिया?", "label": 2, "note": "Presupposes prior lying"},
    {"text": "तुमने अपनी गाड़ी बेचना बंद किया?", "label": 2, "note": "Presupposition: owned a car"},
    # Negation scope
    {"text": "मैंने सभी छात्रों को नहीं पढ़ाया।", "label": 2, "note": "Negation scope: none or not-all?"},
    {"text": "हर आदमी किसी से नहीं डरता।", "label": 2, "note": "Quantifier + negation scope"},
    # Metonymy
    {"text": "संसद ने नया क़ानून पास किया।", "label": 2, "note": "'संसद' = building or members?"},
    {"text": "दिल्ली ने फ़ैसला किया।", "label": 2, "note": "City = government?"},
    # Idiomatic vs literal
    {"text": "उसने आँखें बंद कर लीं।", "label": 2, "note": "Literal eye-closing or ignored?"},
    {"text": "उसने हाथ उठा दिया।", "label": 2, "note": "Raised hand (vote) or gave up?"},
    {"text": "वह पानी में आ गई।", "label": 2, "note": "Literal or 'in trouble'?"},
    {"text": "उसकी नाक कट गई।", "label": 2, "note": "Literal or lost honor/prestige?"},
    # Modal ambiguity
    {"text": "तुम्हें जाना होगा।", "label": 2, "note": "Epistemic vs deontic modal"},
    {"text": "वह घर पर हो सकता है।", "label": 2, "note": "Epistemic possibility"},
    # Double negation / scalar
    {"text": "कोई नहीं आया नहीं।", "label": 2, "note": "Double negation interpretation"},
    {"text": "मुझे थोड़ा नहीं बहुत पसंद है।", "label": 2, "note": "Scalar implicature"},
    # Role ambiguity
    {"text": "डॉक्टर ने नर्स से कहा कि वह ग़लत थी।", "label": 2, "note": "'वह' = doctor or nurse?"},
    {"text": "राजा ने मंत्री से पूछा कि वह कब आएगा।", "label": 2, "note": "'वह' = king or minister?"},
    # More semantic
    {"text": "सभी बच्चों ने किसी एक खिलाड़ी को वोट दिया।", "label": 2, "note": "Scope: same player or each different?"},
    {"text": "हर शिक्षक किसी न किसी छात्र को जानता है।", "label": 2, "note": "Quantifier interaction"},
    {"text": "वह तो बस यही चाहता था।", "label": 2, "note": "Focus particle scope"},
    {"text": "उसने भी वही किया जो उसे कहा था।", "label": 2, "note": "Coreference chain"},
    {"text": "क्या वह सच बोलता है या नहीं?", "label": 2, "note": "Interrogative scope"},
    {"text": "नेताओं ने जनता को जो वादा किया वह नहीं निभाया।", "label": 2, "note": "Relative + negation scope"},
]

df = pd.DataFrame(raw_data)
label_names = {0: 'Lexical', 1: 'Syntactic', 2: 'Semantic'}
df['label_name'] = df['label'].map(label_names)

print(f'Total sentences: {len(df)}')
print('\nClass Distribution:')
print(df['label_name'].value_counts())

# Plot distribution
plt.figure(figsize=(7, 4))
colors = ['#4472C4', '#ED7D31', '#70AD47']
df['label_name'].value_counts().plot(kind='bar', color=colors, edgecolor='white')
plt.title('Hindi Ambiguity Dataset — Class Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Ambiguity Type')
plt.ylabel('Number of Sentences')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Dataset ready!')
df[['text', 'label_name', 'note']].head(10)

## 🔀 Step 3 — Train/Validation/Test Split

In [ ]:
texts = df['text'].tolist()
labels = df['label'].tolist()

# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Train size   : {len(X_train)}')
print(f'Val size     : {len(X_val)}')
print(f'Test size    : {len(X_test)}')

## 🔤 Step 4 — Tokenization with mBERT

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
MAX_LEN = 128

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Test tokenization on a Hindi sentence
sample = "मुझे आम खाना है।"
tokens = tokenizer.tokenize(sample)
print(f'\nSample Hindi sentence : {sample}')
print(f'WordPiece tokens      : {tokens}')
print(f'\n✅ Tokenizer loaded!')

In [ ]:
# ──────────────────────────────────────────
# PyTorch Dataset class
# ──────────────────────────────────────────
class HindiAmbiguityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

BATCH_SIZE = 8

train_dataset = HindiAmbiguityDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = HindiAmbiguityDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = HindiAmbiguityDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print('✅ DataLoaders ready!')

## 🤖 Step 5 — Load mBERT Model

In [ ]:
NUM_CLASSES = 3  # Lexical, Syntactic, Semantic

print(f'Loading model: {MODEL_NAME}')
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print('✅ mBERT model loaded!')

## 🏋️ Step 6 — Training Loop

In [ ]:
# ──────────────────────────────────────────
# Hyperparameters
# ──────────────────────────────────────────
EPOCHS       = 8
LR           = 2e-5
WARMUP_STEPS = 10

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

# ──────────────────────────────────────────
# Helper functions
# ──────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct = 0, 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), acc, f1, all_preds, all_labels

print('✅ Training setup complete. Starting training...')

In [ ]:
# ──────────────────────────────────────────
# TRAINING
# ──────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0

print('=' * 65)
print(f'{"Epoch":^6} | {"Train Loss":^10} | {"Train Acc":^10} | {"Val Loss":^10} | {"Val Acc":^10} | {"Val F1":^8}')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')
        flag = ' ⭐ BEST'
    else:
        flag = ''

    print(f'{epoch:^6} | {train_loss:^10.4f} | {train_acc:^10.4f} | {val_loss:^10.4f} | {val_acc:^10.4f} | {val_f1:^8.4f}{flag}')

print('=' * 65)
print(f'\n✅ Training complete! Best Val F1: {best_val_f1:.4f}')

## 📊 Step 7 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss curves
axes[0].plot(epochs_range, history['train_loss'], 'o-', color='#4472C4', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   's--', color='#ED7D31', label='Val Loss',   linewidth=2)
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy + F1 curves
axes[1].plot(epochs_range, history['train_acc'], 'o-',  color='#4472C4', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'],   's--', color='#ED7D31', label='Val Acc',   linewidth=2)
axes[1].plot(epochs_range, history['val_f1'],    '^:', color='#70AD47', label='Val F1',    linewidth=2)
axes[1].set_title('Training Accuracy & F1 Score', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('mBERT Fine-tuning on Hindi Ambiguity Classification', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved!')

## 🎯 Step 8 — Test Set Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pt', map_location=device))

test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, device)

print('=' * 45)
print(f'  TEST SET RESULTS (Best Model)')
print('=' * 45)
print(f'  Test Loss    : {test_loss:.4f}')
print(f'  Test Accuracy: {test_acc:.4f}  ({test_acc*100:.1f}%)')
print(f'  Macro F1     : {test_f1:.4f}')
print('=' * 45)

class_names = ['Lexical', 'Syntactic', 'Semantic']
print('\nDetailed Classification Report:')
print(classification_report(test_labels, test_preds, target_names=class_names, digits=4))

## 🔲 Step 9 — Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 16, 'weight': 'bold'}
)
plt.title('Confusion Matrix — Hindi Ambiguity Classifier', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

## 🚀 Step 10 — Inference on New Hindi Sentences

You can test the model on **any Hindi sentence** — just change `test_sentences` below.

In [ ]:
# ──────────────────────────────────────────
# Inference function
# ──────────────────────────────────────────
def predict_ambiguity(sentence, model, tokenizer, device, max_len=128):
    """
    Predict the type of ambiguity in a Hindi sentence.
    Returns label name and confidence scores for all 3 classes.
    """
    model.eval()
    label_map = {0: '🔤 Lexical', 1: '🌳 Syntactic', 2: '🔬 Semantic'}

    encoding = tokenizer(
        sentence,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()

    pred_class = probs.argmax()
    return label_map[pred_class], probs


# ──────────────────────────────────────────
# 🔽  CHANGE THESE SENTENCES TO TEST ANYTHING
# ──────────────────────────────────────────
test_sentences = [
    # Lexical
    "वह बाज़ार से आम लेकर आई।",
    "उसका काला जादू डरावना था।",
    "पत्ता नहीं था उसे खेल का।",
    # Syntactic
    "मैंने दूरबीन से पहाड़ पर बैठे आदमी को देखा।",
    "उसने लाठी लेकर आदमी को भगाया।",
    "वह सफेद कपड़े पहनी लड़की का दोस्त था।",
    # Semantic
    "राम ने श्याम को बताया कि वह पास हो गया।",
    "हर बच्चे ने कोई खेल नहीं खेला।",
    "क्या तुमने झूठ बोलना बंद किया?",
    # Your own test sentence:
    "उसका पानी उतर गया बाज़ार में।",
]

print('━' * 80)
print(f'  {"Hindi Sentence":<48} | {"Predicted Type":<16} | Confidence')
print('━' * 80)
for sent in test_sentences:
    pred_label, probs = predict_ambiguity(sent, model, tokenizer, device)
    confidence = probs.max() * 100
    conf_bar = '█' * int(confidence // 10)
    print(f'  {sent[:46]:<48} | {pred_label:<16} | {confidence:.1f}% {conf_bar}')
print('━' * 80)

## 🎨 Step 11 — Attention Visualization (Interpretability)

In [ ]:
def visualize_attention(sentence, model, tokenizer, device, layer=11, head=0):
    """
    Visualize attention weights for a Hindi sentence.
    Shows which tokens attend to which for interpretability.
    """
    model.eval()
    encoding = tokenizer(
        sentence,
        return_tensors='pt',
        truncation=True,
        max_length=64
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True
        )

    # Extract attention weights [layers, heads, seq, seq]
    attentions = outputs.attentions[layer][0, head].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())
    # Remove padding tokens
    real_len = attention_mask[0].sum().item()
    tokens = tokens[:real_len]
    attentions = attentions[:real_len, :real_len]

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        attentions, xticklabels=tokens, yticklabels=tokens,
        cmap='YlOrRd', linewidths=0.3, linecolor='white'
    )
    plt.title(f'Self-Attention Weights — Layer {layer+1}, Head {head+1}', fontsize=13, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig('attention_viz.png', dpi=150, bbox_inches='tight')
    plt.show()

# Show attention for an ambiguous sentence
sample_sent = "मुझे आम खाना है।"
pred, probs = predict_ambiguity(sample_sent, model, tokenizer, device)
print(f'Sentence : {sample_sent}')
print(f'Predicted: {pred}   (Lexical: {probs[0]:.2f}, Syntactic: {probs[1]:.2f}, Semantic: {probs[2]:.2f})')
visualize_attention(sample_sent, model, tokenizer, device)

## 💾 Step 12 — Save the Model

In [ ]:
import os

SAVE_DIR = './hindi_ambiguity_model'
os.makedirs(SAVE_DIR, exist_ok=True)

# Load best checkpoint and save
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f'✅ Model and tokenizer saved to: {SAVE_DIR}')
print('\nFiles saved:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f'  {f:40s}  {size:8.1f} KB')

## 🔁 Step 13 — Load & Re-use Saved Model (Optional)

In [ ]:
# Load from disk and run inference
from transformers import BertTokenizer, BertForSequenceClassification

loaded_tokenizer = BertTokenizer.from_pretrained('./hindi_ambiguity_model')
loaded_model = BertForSequenceClassification.from_pretrained('./hindi_ambiguity_model')
loaded_model = loaded_model.to(device)

# Quick test
sentence = "राम ने श्याम को बताया कि वह फेल हो गया।"
label, scores = predict_ambiguity(sentence, loaded_model, loaded_tokenizer, device)
print(f'Sentence  : {sentence}')
print(f'Prediction: {label}')
print(f'Scores    : Lexical={scores[0]:.3f}  Syntactic={scores[1]:.3f}  Semantic={scores[2]:.3f}')
print('\n✅ Model loaded and working!')

---
## 📋 Summary

| Component | Detail |
|-----------|--------|
| **Model** | `bert-base-multilingual-cased` (mBERT) |
| **Task** | 3-class Hindi Ambiguity Classification |
| **Classes** | Lexical · Syntactic · Semantic |
| **Dataset** | 90 annotated Hindi sentences |
| **Split** | 70% train / 15% val / 15% test |
| **Optimizer** | AdamW with linear warm-up schedule |
| **Key feature** | WordPiece tokenization for Devanagari |

### References
- Devlin et al. (2019) — *BERT: Pre-training of Deep Bidirectional Transformers* 
- Pires et al. (2019) — *How Multilingual is Multilingual BERT?*
- Bhat et al. (2018) — *Universal Dependency Parsing for Hindi*
- HuggingFace `transformers` library — https://huggingface.co/bert-base-multilingual-cased
